# T1.1 — Anti-spoof model smoke test

Verify `mo-thecreator/Deepfake-audio-detection` (a) loads under
200 MB, (b) returns class probabilities on 16 kHz mono float32 input,
(c) separates real speech from SpeechT5-generated speech on a histogram.

Corpus: 10 real clips from LibriSpeech test-clean + 10 SpeechT5 clips
generated in this notebook.

Acceptance (see docs/TASKS.md T1.1): model loads, logits present, real
vs. synthetic histograms show visible separation.

Note: `MelodyMachine/Deepfake-audio-detection-V2` failed smoke test —
predicts all audio as real (p(synth) ≈ 0.0 for all clips). See docs/ARCHITECTURE.md §2.2.

In [ ]:
# --- VishGuard setup cell (copy from notebooks/00_colabSetup.ipynb) ---
import os, sys, subprocess
from pathlib import Path
REPO_URL = os.environ.get('VISHGUARD_REPO_URL', 'https://github.com/ben-blake/vishguard.git')
DRIVE_SRC = '/content/drive/MyDrive/vishguard'
REPO_DIR = Path('/content/vishguard')
ON_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
def sh(cmd, check=True):
    print('$', cmd); subprocess.run(cmd, shell=True, check=check)
if ON_COLAB:
    try:
        from google.colab import drive; drive.mount('/content/drive', force_remount=False)
    except Exception as e: print('Drive mount skipped:', e)
    if REPO_URL:
        if REPO_DIR.exists(): sh(f'cd {REPO_DIR} && git pull --ff-only')
        else: sh(f'git clone {REPO_URL} {REPO_DIR}')
    elif Path(DRIVE_SRC).exists():
        REPO_DIR.mkdir(parents=True, exist_ok=True)
        sh(f'rsync -a --delete --exclude .venv --exclude __pycache__ {DRIVE_SRC}/ {REPO_DIR}/')
    else:
        raise RuntimeError('Set VISHGUARD_REPO_URL or copy the repo to MyDrive/vishguard/')
    sh(f'pip install -q -e {REPO_DIR}')
    sh(f'pip install -q -r {REPO_DIR}/requirements.txt')
    src = str(REPO_DIR / 'src')
    if src not in sys.path: sys.path.insert(0, src)
import vishguard; print('vishguard', vishguard.__version__)

In [ ]:
import numpy as np
import torch
from transformers import pipeline

# Confirmed working model (ARCHITECTURE.md §2.2). MelodyMachine V2 failed smoke test.
SPOOF_MODEL_ID = 'mo-thecreator/Deepfake-audio-detection'
ALT_A_MODEL_ID  = 'MelodyMachine/Deepfake-audio-detection-V2'  # FAILED — all scores 0.0

device = 0 if torch.cuda.is_available() else -1
spoof = pipeline('audio-classification', model=SPOOF_MODEL_ID, device=device)
print('Primary labels:', spoof.model.config.id2label)
params = sum(p.numel() for p in spoof.model.parameters())
print(f'params={params/1e6:.1f}M  fp32-size={(params*4)/1e6:.0f} MB')

In [ ]:
# Real speech: 10 LibriSpeech test-clean clips via HF datasets (streaming).
from datasets import load_dataset
import librosa

real_clips = []
ds = load_dataset('openslr/librispeech_asr', 'clean', split='test', streaming=True,
                  trust_remote_code=True)
for i, ex in enumerate(ds):
    if i >= 10: break
    audio = ex['audio']
    samples = audio['array'].astype(np.float32)
    sr = audio['sampling_rate']
    if sr != 16_000:
        samples = librosa.resample(samples, orig_sr=sr, target_sr=16_000)
    real_clips.append(samples)
print(f'real_clips: {len(real_clips)} (sample shapes: {[c.shape for c in real_clips[:3]]})')

In [ ]:
# Synthetic speech: 10 SpeechT5 clips (also reused for the T3.1 eval corpus).
# cmu-arctic-xvectors stores embeddings in spkrec-xvect.zip — extract directly.
import pathlib, zipfile, numpy as np
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan
from huggingface_hub import snapshot_download

tts_proc = SpeechT5Processor.from_pretrained('microsoft/speecht5_tts')
tts_model = SpeechT5ForTextToSpeech.from_pretrained('microsoft/speecht5_tts')
vocoder = SpeechT5HifiGan.from_pretrained('microsoft/speecht5_hifigan')

xvec_repo = snapshot_download(
    repo_id='Matthijs/cmu-arctic-xvectors',
    repo_type='dataset',
    ignore_patterns=['*.py'],
)
zip_path = pathlib.Path(xvec_repo) / 'spkrec-xvect.zip'
extract_dir = pathlib.Path('/tmp/cmu_arctic_xvect')
extract_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(extract_dir)

npy_files = sorted(extract_dir.rglob('*.npy'))
pt_files  = sorted(extract_dir.rglob('*.pt'))

if npy_files:
    # Each .npy is one speaker embedding (shape 512,).
    xvec = np.load(npy_files[7306 % len(npy_files)])
    speaker = torch.tensor(xvec).unsqueeze(0)
elif pt_files:
    data = torch.load(pt_files[0], map_location='cpu')
    if isinstance(data, torch.Tensor):
        speaker = data[min(7306, data.shape[0] - 1)].unsqueeze(0)
    elif isinstance(data, dict):
        speaker = torch.stack(list(data.values()))[0].unsqueeze(0)
    else:
        raise ValueError(f'unexpected pt type: {type(data)}')
else:
    print('WARNING: using normalized random embedding (fine for smoke test)')
    speaker = torch.nn.functional.normalize(torch.randn(1, 512), dim=-1)
print(f'speaker shape: {speaker.shape}')

scripts = [
    'Hello, this is an automated message from your bank.',
    'We detected unusual activity on your account. Please confirm your identity.',
    'Your package delivery has been rescheduled. Press one to continue.',
    'This is the IRS. You have an outstanding tax balance.',
    'Your computer has been compromised. Do not turn it off.',
    'Congratulations, you have won a five hundred dollar gift card.',
    'This is your car warranty specialist calling.',
    'Please verify the last four digits of your social security number.',
    'Your subscription will renew automatically tomorrow.',
    'Press one to speak with a representative about your loan.',
]
synth_clips = []
for text in scripts:
    inputs = tts_proc(text=text, return_tensors='pt')
    with torch.no_grad():
        wav = tts_model.generate_speech(inputs['input_ids'], speaker, vocoder=vocoder)
    synth_clips.append(wav.numpy().astype(np.float32))
print(f'synth_clips: {len(synth_clips)} (shapes: {[c.shape for c in synth_clips[:3]]})')

In [ ]:
# Run detector on both sets. Capture pSynthetic per clip.
# NOTE: if scores are all near-zero, swap SPOOF_MODEL_ID and reset _fake_label_cache.
_fake_label_cache = None

def _find_fake_label(id2label):
    real_kw = ('real', 'genuine', 'bonafide', 'bona', 'human', 'natural', 'label_0')
    fake_kw = ('fake', 'spoof', 'synthetic', 'deepfake', 'generated', 'ai', 'tts', 'label_1')
    for lname in id2label.values():
        if any(k in lname.lower() for k in fake_kw):
            return lname
    if len(id2label) == 2:
        for lname in id2label.values():
            if not any(k in lname.lower() for k in real_kw):
                return lname
    return id2label[max(id2label.keys())]

def p_synthetic(clip):
    global _fake_label_cache
    preds = spoof({'array': clip, 'sampling_rate': 16_000}, top_k=None)
    if _fake_label_cache is None:
        _fake_label_cache = _find_fake_label(spoof.model.config.id2label)
        print(f'id2label  : {spoof.model.config.id2label}')
        print(f'fake_label: "{_fake_label_cache}"')
        print(f'sample preds (first clip): {preds}')
    for p in preds:
        if p['label'] == _fake_label_cache:
            return p['score']
    return 0.0

real_scores  = [p_synthetic(c) for c in real_clips]
synth_scores = [p_synthetic(c) for c in synth_clips]
# Show full precision so near-zero scores are visible
print('real   p(synth):', [f'{s:.4f}' for s in real_scores])
print('synth  p(synth):', [f'{s:.4f}' for s in synth_scores])
print(f'mean real={sum(real_scores)/len(real_scores):.4f}  mean synth={sum(synth_scores)/len(synth_scores):.4f}')
print()
print('--- To switch to Alt A, run in a new cell:')
print('spoof = pipeline("audio-classification", model=ALT_A_MODEL_ID, device=device)')
print('_fake_label_cache = None  # reset label cache')

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 4))
plt.hist(real_scores, bins=10, alpha=0.6, label='real speech (LibriSpeech)')
plt.hist(synth_scores, bins=10, alpha=0.6, label='SpeechT5 synthetic')
plt.axvline(0.5, color='k', ls='--', lw=1)
plt.xlabel('p(synthetic)'); plt.ylabel('count')
plt.title('T1.1 — mo-thecreator/Deepfake-audio-detection')
plt.legend(); plt.tight_layout(); plt.show()
import numpy as np
print(f'mean real={np.mean(real_scores):.3f}  mean synth={np.mean(synth_scores):.3f}')
print(f'separation at 0.5 threshold: real<0.5={sum(s<0.5 for s in real_scores)}/10, synth>=0.5={sum(s>=0.5 for s in synth_scores)}/10')

## Acceptance checklist

- [x] Model loads and prints id2label  (`{0: 'real', 1: 'fake'}` for mo-thecreator)
- [ ] Params under ~200 MB fp32
- [x] `p_synthetic` defined in [0, 1] for all 20 clips
- [ ] Histograms visibly separated; record means and threshold-0.5 counts

**T1.1 results (Colab run 2026-04-21):**
- real p(synth):  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
- synth p(synth): [0.002, 0.0, 0.121, 0.992, 1.0, 0.998, 0.999, 0.003, 0.954, 0.988]
- Threshold-0.5: real<0.5 = 10/10, synth>=0.5 = 7/10 — separation visible.

If histograms are flat, fallback to a zero-shot audio classifier or
escalate to a model trained on ElevenLabs/RVC synthetics.